# Day02 Practice · PC-01~PC-11 공통 문제

각 cluster는 setup 셀 뒤에 독립적으로 시도할 수 있음.  
공통 문제 셀은 Q와 A에서 동일하며, 완성 답과 실제 정답 출력은 Practice A의 별도 셀에만 있음.

In [ ]:
import copy

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.set_printoptions(precision=6, sci_mode=False)

## PC-01 · `w,b` 두 parameter의 책임

목적: gradient 계산, update, fresh evidence, reset을 서로 다른 책임으로 배치함.

아래 TODO 세 구역을 완성하고 다음을 확인함.

- update 뒤 `w,b`가 모두 바뀌고 새 prediction/loss가 계산됨
- update만으로 `.grad`가 사라지지 않음
- reset 뒤 두 `.grad`가 모두 0임

In [ ]:
p01_w = torch.tensor([1.0], requires_grad=True)
p01_b = torch.tensor([0.0], requires_grad=True)
p01_x = torch.tensor([2.0])
p01_target = torch.tensor([5.0])
p01_loss = ((p01_w * p01_x + p01_b - p01_target) ** 2).mean()

# TODO 1: gradient 계산
# TODO 2: torch.no_grad() 안에서 w,b update
p01_fresh_prediction = p01_w * p01_x + p01_b
p01_fresh_loss = ((p01_fresh_prediction - p01_target) ** 2).mean()
p01_grad_before_reset = None if p01_w.grad is None else (p01_w.grad.item(), p01_b.grad.item())
# TODO 3: fresh evidence 확인 뒤 w.grad와 b.grad reset

p01_grad_ready = p01_w.grad is not None and p01_b.grad is not None
p01_reset_ready = p01_grad_ready and p01_w.grad.item() == 0.0 and p01_b.grad.item() == 0.0
p01_values_changed = p01_w.item() != 1.0 and p01_b.item() != 0.0
print("gradient 계산:", p01_grad_ready)
print("두 값 update:", p01_values_changed)
print("fresh prediction / loss:", p01_fresh_prediction.item(), p01_fresh_loss.item())
print("reset 전 grad:", p01_grad_before_reset)
print("두 grad reset:", p01_reset_ready)

<details><summary>Hint 1 · 사고 방향</summary>

`계산 → 사용 → 정리` 순서로 세 구역을 나눔.
</details>

<details><summary>Hint 2 · API</summary>

`backward()`, `torch.no_grad()`, `zero_()`를 사용함.
</details>

## PC-02 · 학습할 값을 Module에 등록함

목적: `w,b`를 등록된 parameter로 만들고 같은 affine forward를 유지함.

완료 조건

- `named_parameters()` 이름이 `w,b`임
- 입력 2의 forward 결과 shape가 `[1]`임

In [ ]:
class P02ManagedAffine(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO 1: w=1.6, b=0.3을 nn.Parameter로 등록
        self.w = torch.tensor([1.6])
        self.b = torch.tensor([0.3])

    def forward(self, value):
        # TODO 2: w*x+b 반환
        return value

p02_model = P02ManagedAffine()
p02_names = [name for name, _ in p02_model.named_parameters()]
p02_output = p02_model(torch.tensor([2.0]))
print("registered names:", p02_names)
print("output shape:", tuple(p02_output.shape))
print("등록 완료:", p02_names == ["w", "b"])

<details><summary>Hint 1 · 사고 방향</summary>

Module이 모아야 할 것은 입력이 아니라 학습으로 바뀌는 상태임.
</details>

<details><summary>Hint 2 · API</summary>

`nn.Parameter(torch.tensor([...]))`와 `self.w * value + self.b`를 사용함.
</details>

## PC-03 · Linear boundary와 XOR

목적: 한 직선으로 분리 가능한 패턴과 불가능한 패턴을 근거로 구분함.

준비된 후보 두 개만 시험하고, 결과 아래에 XOR이 어려운 이유를 한 문장으로 기록함.

In [ ]:
p03_x = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
p03_target = torch.tensor([0, 1, 1, 0])
p03_trials = [
    ("후보 A", [1., 1.], -0.5),
    ("후보 B", [1., -1.], -0.5),
]

for label, weight, bias in p03_trials:
    p03_scores = p03_x @ torch.tensor(weight).reshape(2, 1) + bias
    p03_decisions = (p03_scores >= 0).to(torch.int64).squeeze(1)
    p03_accuracy = (p03_decisions == p03_target).float().mean().item()
    print(label, p03_decisions.tolist(), p03_accuracy)

print("확인한 후보 수:", len(p03_trials))

### 한 문장 설명

`XOR은 ________________________________________________ 때문에 하나의 Linear 경계로 나누기 어려움.`

완료 조건: 두 후보의 네 판정과 accuracy를 확인하고, 점의 배치를 근거로 설명함.

<details><summary>Hint 1 · 사고 방향</summary>

0 클래스 두 점과 1 클래스 두 점이 각각 어디에 놓이는지 그림에서 확인함.
</details>

<details><summary>Hint 2 · API</summary>

예: weight 두 값과 bias 하나를 Tensor 연산에 넣고 `scores >= 0`을 판정으로 사용함.
</details>

## PC-04 · Linear 사이에 ReLU 넣기

목적: hidden Linear와 output Linear 사이의 비선형성을 코드에서 찾고 XOR 학습 구조를 완성함.

완료 조건

- `forward()`에서 ReLU 위치 확인
- MSE target shape `[4,1]` 확인
- 500 step 뒤 네 XOR 판정 확인

In [ ]:
p04_x = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
p04_target = torch.tensor([[0.], [1.], [1.], [0.]])

class P04XORMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 16)
        self.output = nn.Linear(16, 1)

    def forward(self, value):
        hidden_value = self.hidden(value)
        # TODO 1: hidden_value에 ReLU 적용
        activated_value = hidden_value
        return self.output(activated_value)

p04_ready = False  # TODO 2: forward 수정 뒤 True
if p04_ready:
    torch.manual_seed(42)
    p04_model = P04XORMLP()
    for _ in range(500):
        p04_output = p04_model(p04_x)
        p04_loss = ((p04_output - p04_target) ** 2).mean()
        p04_loss.backward()
        with torch.no_grad():
            for parameter in p04_model.parameters():
                parameter -= 0.1 * parameter.grad
                parameter.grad.zero_()
    with torch.no_grad():
        p04_output = p04_model(p04_x)
        p04_loss = ((p04_output - p04_target) ** 2).mean()
        p04_decisions = (p04_output >= 0.5).to(torch.int64).squeeze(1)
    print("target shape:", tuple(p04_target.shape))
    print("decisions:", p04_decisions.tolist())
    print("MSE:", f"{p04_loss.item():.10f}")
else:
    print("forward에 ReLU를 넣고 p04_ready를 True로 바꾸면 준비된 loop가 실행됨")

<details><summary>Hint 1 · 사고 방향</summary>

두 Linear 사이에서 음수와 양수를 같은 규칙으로 통과시키지 않는 변환이 필요함.
</details>

<details><summary>Hint 2 · API</summary>

`torch.relu(...)`; 각 step은 `forward → MSE → backward → no_grad update → grad zero` 순서임.
</details>

## PC-05 · class별 원점수와 loss 계약

**왜:** 실행 가능한 조합과 우리가 채택한 학습 계약을 구분하고, `2→16→2` 모델의 실제 parameter shape를 읽기 위함.

**완료 조건**
- output `[N,2]`, target `[N]` integer 계약을 설명함
- raw logits를 CrossEntropyLoss에 전달함
- 두 Linear의 weight/bias를 모두 세어 82를 검증함
- Softmax-before-CE가 “실행 불가”라고 쓰지 않음

In [ ]:
class P05Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO 1: Linear(2,16), ReLU, Linear(16,2)를 준비
        self.hidden = nn.Linear(2, 16)
        self.output = nn.Linear(16, 2)

    def forward(self, value):
        # TODO 2: 두 Linear 사이에 ReLU를 넣고 raw logits 반환
        return self.output(self.hidden(value))

p05_model = P05Classifier()
p05_x = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
p05_target = torch.tensor([0, 1, 1, 0], dtype=torch.int64)
p05_logits = p05_model(p05_x)
p05_total = sum(p.numel() for p in p05_model.parameters())
print("logits / target:", tuple(p05_logits.shape), tuple(p05_target.shape), p05_target.dtype)
print("parameter count:", p05_total)

### learner area

- CrossEntropyLoss에 전달할 model output: `________________`
- Softmax를 사용할 수 있는 해석 시점: `________________`
- 82 계산: `________________`

<details><summary>Hint 1 · 사고 방향</summary>

한 sample의 class 후보 수와 target에 기록되는 class 번호 수를 먼저 구분함.
</details>

<details><summary>Hint 2 · API</summary>

`torch.relu`, `nn.CrossEntropyLoss`, `named_parameters()`의 shape와 `numel()`을 확인함.
</details>

## PC-06 · optimizer 책임과 canonical loop

**왜:** gradient 계산, 값 변경, gradient clear를 한 “자동 학습” 동작으로 뭉치지 않기 위함.

**완료 조건**
- `zero_grad → forward → loss → backward → step`을 책임 이유와 함께 배열함
- `step()` 전후 grad와 `zero_grad()` 뒤 상태를 관찰함

In [ ]:
p06_model = nn.Linear(1, 1, bias=False)
p06_optimizer = torch.optim.SGD(p06_model.parameters(), lr=0.1)
p06_x = torch.tensor([[2.0]])
p06_target = torch.tensor([[0.0]])

# learner area: 아래 역할 카드를 실제 API 순서로 바꾸고 한 step을 완성
p06_roles = ["지난 흔적 정리", "예측", "오차", "gradient 계산", "값 변경"]
p06_ready = False
if p06_ready:
    # TODO: canonical five responsibilities
    pass
else:
    print("역할 카드:", p06_roles)

### learner area

`_____ → _____ → _____ → _____ → _____`

`step()` 뒤 grad 예상과 관찰: `________________`

<details><summary>Hint 1 · 사고 방향</summary>

이전 step의 흔적을 먼저 정리하고, 현재 loss가 만든 gradient를 마지막 값 변경에서 사용함.
</details>

<details><summary>Hint 2 · API</summary>

`optimizer.zero_grad()`, model call, criterion, `loss.backward()`, `optimizer.step()`을 사용함.
</details>

## PC-07 · learning rate와 SGD/Adam evidence 읽기

**왜:** optimizer 이름이나 마지막 loss 하나만 보고 보편적 승자를 선언하지 않기 위함.

**완료 조건**
- 같은 data/model/init/200-step 측정 조건을 적음
- 네 curve의 1/10/50/100/200 step을 근거로 비교함
- Adam이 gradient와 gradient 제곱의 running average를 함께 쓴다는 beginner-safe 직관을 설명함
- 현재 조건 밖에서 말할 수 없는 것을 한 문장으로 씀

In [ ]:
# learner area: 같은 초기 state를 복사해 네 조건을 200 step 비교
p07_conditions = {
    "SGD-small": None,   # TODO: optimizer와 lr
    "SGD-large": None,
    "Adam-small": None,
    "Adam-large": None,
}
print("채울 조건:", list(p07_conditions))
print("checkpoint:", [1, 10, 50, 100, 200])

### learner area

- 같게 둔 조건: `________________`
- 곡선에서 직접 본 근거: `________________`
- Adam의 두 running-average 직관: `________________`
- 제한된 결론: `________________`

<details><summary>Hint 1 · 사고 방향</summary>

“무엇을 고정했는가 / 무엇을 바꿨는가 / 이 범위 밖에서 말할 수 없는가”의 세 문장으로 나눔.
</details>

<details><summary>Hint 2 · API와 직관</summary>

동일한 `state_dict`를 load한 model마다 SGD 또는 Adam과 lr을 하나씩 지정함. Adam 설명에서는 최근 gradient 방향의 평균과 gradient 제곱의 평균이 각각 어떤 정보를 남기는지 구분함.
</details>

## PC-08 · Dataset, DataLoader, current batch, device

**왜:** 저장된 example과 batch 전달의 책임을 분리하고, 현재 prediction을 현재 target과 연결하기 위함.

**완료 조건**
- 500 samples, batch_size 64, 8 batches, last 52를 확인함
- loop 안에서 current input/target을 함께 사용함
- model/input/target의 same-device invariant를 설명함

In [ ]:
p08_x = torch.arange(1000, dtype=torch.float32).reshape(500, 2)
p08_y = torch.arange(500, dtype=torch.int64) % 2
# TODO 1: TensorDataset 생성
# TODO 2: batch_size=64, shuffle=False, drop_last=False DataLoader 생성
p08_ready = False
if p08_ready:
    p08_sizes = [len(batch_x) for batch_x, batch_y in p08_loader]
    print(len(p08_dataset), len(p08_loader), p08_sizes)
else:
    print("500개 input/target 준비 완료")

### learner area

- Dataset 책임: `________________`
- DataLoader 책임: `________________`
- current batch target discipline: `________________`
- 같은 device에 둘 대상: `________________`

<details><summary>Hint 1 · 사고 방향</summary>

마지막 묶음은 `500 % 64`를 직접 계산하기보다 실제 iteration의 첫 dimension을 기록함.
</details>

<details><summary>Hint 2 · API</summary>

`TensorDataset(p08_x, p08_y)`, `DataLoader(..., batch_size=64, drop_last=False)`를 사용함.
</details>

## PC-09 · 평가, accuracy, full pipeline, shape safety

**왜:** model mode와 gradient recording을 구분하고, scalar loss 뒤에 숨은 잘못된 shape를 찾기 위함.

**완료 조건**
- `eval()`과 `no_grad()`의 책임을 각각 적음
- logits→argmax→correct/total accuracy를 현재 batch target으로 계산함
- `[n,1]`과 `[n]`의 expanded shape를 찾아 수정함
- shape/dtype/device guard를 한 줄 이상 둠

In [ ]:
p09_predictions = torch.tensor([[0.1], [0.9], [0.8], [0.2]])
p09_targets = torch.tensor([0.0, 1.0, 1.0, 0.0])
print("현재 shape:", tuple(p09_predictions.shape), tuple(p09_targets.shape))
# learner area: 두 Tensor를 빼서 실제 result shape를 확인하고 pairwise [4,1]로 수정

def p09_evaluate(model, loader, device):
    # TODO: eval + no_grad + current batch accuracy
    return None

### learner area

- model mode switch: `________________`
- gradient recording switch: `________________`
- accuracy numerator / denominator: `________________`
- broadcasting 최소 수정: `________________`

<details><summary>Hint 1 · 사고 방향</summary>

모델의 동작 상태와 연산 기록 여부를 서로 다른 두 질문으로 나눔. Accuracy는 rate보다 count를 먼저 누적함.
</details>

<details><summary>Hint 2 · API</summary>

`model.eval()`, `with torch.no_grad():`, `argmax(dim=1)`, `reshape(-1,1)`을 확인함.
</details>

## PC-10 · debugging과 Sprint readiness 통합

**왜:** 예외 문구 암기 대신 기대 계약과 실제 관찰을 비교하고, 낯선 코드의 역할을 재구성하기 위함.

**완료 조건**
- 실제 guarded failure 하나에서 원인/경계/복구를 기록함
- Softmax-before-CE와 grad persistence 같은 non-exception bug를 구분함
- assessment 답을 복사하지 않고 data→model→loss→gradient→update→evaluation 역할을 찾음
- CNN/RNN bridge는 “다음에 필요한 구조” 질문까지만 답함

In [ ]:
p10_model = nn.Linear(2, 1)
p10_bad_input = torch.ones(1, 2, dtype=torch.float64)
try:
    p10_model(p10_bad_input)
except (RuntimeError, TypeError) as p10_error:
    print("observed type:", type(p10_error).__name__)
    print("observed message:", str(p10_error).splitlines()[0])

# learner area: expected dtype, mismatch, minimal correction, verify를 기록하고 실행

### learner area

| Expected | Observed | Mismatch | Minimal repair | Verify |
|---|---|---|---|---|
|  |  |  |  |  |

- 예외가 없어도 확인할 semantic bug: `________________`
- CNN bridge 질문: `________________`
- RNN bridge 질문: `________________`

<details><summary>Hint 1 · 사고 방향</summary>

오류가 난 줄보다 먼저 model weight dtype과 input dtype이라는 계약을 비교함.
</details>

<details><summary>Hint 2 · API</summary>

`p10_bad_input.to(dtype=p10_model.weight.dtype)`로 최소 수정 뒤 output shape를 검증함.
</details>

## PC-11 · 누적 micro-retrieval과 clean Run All

**왜:** Day1을 다시 배우지 않고 새 숫자에서 필요한 연산을 직접 선택하고, 새 kernel에서도 같은 결과를 만들기 위함.

**완료 조건**
- Tensor 생성/reshape, indexing/slicing, squeeze/unsqueeze를 직접 완성함
- cat/stack과 `*`/matmul을 결과 목적에 맞게 선택함
- `.long()`/`.float()`의 dtype과 소수부 truncation을 확인함
- Sigmoid를 직접 적용해 입력 0의 출력을 관찰함
- manual MSE와 새로 만든 `nn.MSELoss()` 결과의 parity를 확인함
- import, seed, 숨은 변수, Restart Kernel → Run All 네 항목을 체크함

연습 숫자는 앞의 시범과 다르게 구성했으며, 결과를 외우지 않고 필요한 연산을 다시 선택함.

In [ ]:
# 한 셀 안에서 독립적으로 완성함. None은 untouched Run All을 안전하게 유지함.
p11_seed = None              # TODO: 비교 가능한 실행을 위한 정수 seed
p11_grid = None              # TODO: 0~17을 float32 Tensor [3,6]으로 구성
p11_row = None               # TODO: 두 번째 row 선택
p11_slice = None             # TODO: row 1~2와 column 2~4 범위 선택

p11_with_axis = torch.arange(6, dtype=torch.float32).reshape(3, 1, 2)
p11_squeezed = None          # TODO: 가운데 size-1 axis 제거
p11_column = None            # TODO: 마지막에 size-1 axis 추가

p11_a = torch.arange(6, dtype=torch.float32).reshape(3, 2)
p11_b = p11_a + 10
p11_cat = None               # TODO: 기존 row axis를 이어 붙임
p11_stack = None             # TODO: 두 Tensor를 새 axis 1에 쌓음
p11_elementwise = None       # TODO: p11_a와 p11_b의 같은 위치끼리 곱함
p11_weight = torch.tensor([[1., 0., -1.], [0., 1., 1.]])
p11_matmul = None            # TODO: [3,2]와 [2,3]의 행렬곱

p11_decimal = torch.tensor([4.8, -1.6, 0.2])
p11_long = None              # TODO: int64로 변환
p11_float = None             # TODO: 다시 float32로 변환
p11_sigmoid = None           # TODO: [-3,0,3]에 Sigmoid 직접 적용

p11_prediction = torch.tensor([1.0, 2.0, 4.0])
p11_target = torch.tensor([0.0, 2.0, 5.0])
p11_mse_manual = None        # TODO: 오차 제곱의 평균
p11_mse_builtin = None       # TODO: nn.MSELoss()를 새로 만들어 계산

p11_required = [p11_grid, p11_row, p11_slice, p11_squeezed, p11_column,
                p11_cat, p11_stack, p11_elementwise, p11_matmul,
                p11_long, p11_float, p11_sigmoid, p11_mse_manual, p11_mse_builtin]
if p11_seed is not None and all(isinstance(value, torch.Tensor) for value in p11_required):
    print("shape checks:", p11_grid.shape, p11_slice.shape, p11_squeezed.shape, p11_column.shape)
    print("combine checks:", p11_cat.shape, p11_stack.shape)
    print("multiply checks:", p11_elementwise.shape, p11_matmul.shape)
    print("dtype / cast:", p11_decimal.dtype, p11_long.dtype, p11_float.dtype, p11_long)
    print("sigmoid center:", p11_sigmoid[1].item())
    print("MSE parity:", p11_mse_manual.item(), p11_mse_builtin.item())

### learner area · 실행 계약

- 필요한 import가 위에 있음: `[ ]`
- seed를 셀 안에서 고정함: `[ ]`
- 수동 실행으로만 생긴 변수가 없음: `[ ]`
- Restart Kernel → Run All로 위에서 아래까지 통과함: `[ ]`

- `.long()` 뒤 값과 주의점: `________________`
- `cat`/`stack`, `*`/matmul 선택 근거: `________________`
- manual/built-in MSE가 같은 이유: `________________`

<details><summary>Hint 1 · 연산 선택</summary>

값 일부 선택 / 원소 수를 유지한 모양 변경 / 기존 축 연결 / 새 축 생성 / 같은 위치 곱 / 안쪽 축 연결을 먼저 구분함.
</details>

<details><summary>Hint 2 · API</summary>

`torch.arange`, `reshape`, indexing/slicing, `squeeze`, `unsqueeze`, `torch.cat`, `torch.stack`, `*`, `@`/`torch.matmul`, `.long()`, `.float()`, `torch.sigmoid`, `nn.MSELoss()` 중 필요한 것을 고름.
</details>

## Practice Q 종료

답 확인은 `DAY02_PRACTICE_A.ipynb`의 같은 PC 번호에서 수행함. Q에서 작성한 해석은 답안의 숫자보다 먼저 보존함. 마지막으로 Restart Kernel → Run All로 위에서 아래 실행을 확인함.